# Task 1 — Direct Clarity Classification

Fine-tuning **Llama 3.1 8B-Instruct** with 4-bit QLoRA + DoRA to directly classify the clarity of a political answer into three macro-categories: **Ambivalent**, **Clear Reply**, and **Clear Non-Reply**.


In [ ]:
# ============================================================
# CELL 1 — Environment setup
# ============================================================

import os, sys, json, gc
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from huggingface_hub import login
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from datasets import load_from_disk
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

try:
    import yaml
except ImportError as exc:
    raise ImportError("Missing dependency: PyYAML. Install with `pip install pyyaml`.") from exc

# Path configuration and environment detection (Colab / local)
try:
    import google.colab
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    # Base paths on Google Drive
    BASE_DIR = "/content/drive/MyDrive/progettoLLM"
    REPO_DIR = os.path.join(BASE_DIR, "CLARITY")
    hf_cache_dir = os.path.join(BASE_DIR, "hf_cache")
    
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Login HF tramite Colab Secrets
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print(f"Ambiente Google Colab configurato. Cache: {hf_cache_dir}")

except ImportError:
    # Percorsi base in locale
    BASE_DIR = ".."
    REPO_DIR = BASE_DIR
    
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # Lettura del token da file .env locale (cerca prima in CLARITY, poi nella root)
    env_path = os.path.join(REPO_DIR, ".env")
    if not os.path.exists(env_path):
        env_path = ".env"
        
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    os.environ['HF_TOKEN'] = hf_token
                    login(token=hf_token)
                    break
    print("Ambiente locale rilevato.")

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# CELL 2 — Variables and prompt
# ============================================================

# Config from file
CONFIG_PATH = os.path.join(REPO_DIR, "config", "config.yml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f) or {}

# Control flag
RUN_TRAINING = bool(config.get("training", {}).get("enabled", True))

# Model and paths
MODEL_ID = config.get("model", {}).get("id", "meta-llama/Llama-3.2-3B-Instruct")

# Base results directory
RES_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("results_dir", "results"))

TASK1_DIR = os.path.join(REPO_DIR, config.get("paths", {}).get("task1_dir", "results/task1"))

# Dynamically built task-specific paths
OUTPUT_DIR = os.path.join(TASK1_DIR, "checkpoints")
SAVED_MODEL_PATH = os.path.join(TASK1_DIR, "modello_finale")

# Training params (Task 1)
TRAINING_CONFIG = config.get("training", {}).get("task1", {})
SEED = config.get("training", {}).get("seed", 42)
PER_DEVICE_TRAIN_BATCH_SIZE = TRAINING_CONFIG.get("batch_size", 2)
GRADIENT_ACCUMULATION_STEPS = TRAINING_CONFIG.get("gradient_accumulation_steps", 4)
NUM_TRAIN_EPOCHS = TRAINING_CONFIG.get("num_train_epochs", 1)

# Dataset/augmentation config
DATASET_CONFIG = config.get("dataset", {})
DATASET_USE_AUGMENTED = bool(DATASET_CONFIG.get("use_augmented", False))
AUGMENTATION_NAME = DATASET_CONFIG.get("augmentation", {}).get("name", "none")
AUGMENTATION_PARAMS = DATASET_CONFIG.get("augmentation", {}).get("params", {})

# Precision: bfloat16 when supported, otherwise float16
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

# System prompt for direct three-class classification
SYSTEM_PROMPT_TASK1 = """You are an expert political discourse analyst. Classify the politician's answer to a direct question into exactly one of the following 3 categories:

1. 'Clear Reply': The answer has one clear interpretation.
2. 'Ambivalent': The answer is provided, but it is vague or structured in a way that allows multiple interpretations.
3. 'Clear Non-Reply': The respondent openly refuses to answer or states that they cannot provide the requested information.

Reply ONLY with the exact category name (Clear Reply, Ambivalent, or Clear Non-Reply). Do not add any other text."""

# Labels in report order
LABELS = ["Ambivalent", "Clear Reply", "Clear Non-Reply"]

print("Configuration loaded.")

In [ ]:
# ============================================================
# CELL 3 — Dataset management
# ============================================================

from datasets import DatasetDict
from src.data.augmentation import get_augmentation_fn
from src.data.label_utils import build_label_maps

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load train and test splits using REPO_DIR
TRAIN_DATA_PATH = os.path.join(REPO_DIR, DATASET_CONFIG.get("train_path", "dataset/QEvasion/train"))
TEST_DATA_PATH = os.path.join(REPO_DIR, DATASET_CONFIG.get("test_path", "dataset/QEvasion/test"))

train_dataset = load_from_disk(TRAIN_DATA_PATH)
test_dataset  = load_from_disk(TEST_DATA_PATH)
print(f"Train: {len(train_dataset)} examples | Test: {len(test_dataset)} examples")

if DATASET_USE_AUGMENTED:
    if "evasion_label" not in train_dataset.column_names:
        print("Augmentation skipped: 'evasion_label' not found in train split.")
    else:
        ds = DatasetDict({"train": train_dataset, "validation": test_dataset, "test": test_dataset})
        label2id, _ = build_label_maps(ds)
        augment_fn = get_augmentation_fn(AUGMENTATION_NAME)
        ds = augment_fn(ds, label2id, **AUGMENTATION_PARAMS)
        train_dataset = ds["train"]
        print(f"Augmentation applied: {AUGMENTATION_NAME}")

def format_prompt_task1(example):
    """Format one example with the Llama 3.1 chat template for Task 1."""
    question = example.get('interview_question', '')
    answer = example.get('interview_answer', '')
    label    = example.get('clarity_label', '')

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT_TASK1},
        {"role": "user", "content": f"Question: {question}\nPolitician answer: {answer}"},
        {"role": "assistant", "content": str(label)},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Format the training set
formatted_train = train_dataset.map(format_prompt_task1, remove_columns=train_dataset.column_names)

print("\n--- Formatted prompt example ---")
print(formatted_train[0]["text"][:500] + "\n...")

In [ ]:
# ============================================================
# CELL 4 — Model training or loading
# ============================================================

# Clear VRAM before loading the model
torch.cuda.empty_cache()
gc.collect()

# Shared 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

if RUN_TRAINING:
    # ── TRAINING ──────────────────────────────────────────────
    print("Training mode enabled.")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )
    model = prepare_model_for_kbit_training(model)

    # DoRA configuration for Llama attention layers
    peft_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", use_dora=False,
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        optim="paged_adamw_8bit",
        save_strategy="epoch",
        logging_steps=10,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=(COMPUTE_DTYPE == torch.float16),
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        max_grad_norm=0.3,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_steps=100,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="none",
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_train,
        args=training_args,
        peft_config=None,          # modello già wrappato con PEFT
        processing_class=tokenizer,
    )

    print("Starting training...")
    trainer.train()

    # Save adapters and tokenizer
    trainer.model.save_pretrained(SAVED_MODEL_PATH)
    tokenizer.save_pretrained(SAVED_MODEL_PATH)
    print(f"Model saved to: {SAVED_MODEL_PATH}")

else:
    # ── CARICAMENTO DA CHECKPOINT ─────────────────────────────
    print(f"Loading base model ({MODEL_ID}) in 4-bit...")

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=bnb_config, torch_dtype=COMPUTE_DTYPE,
    )

    print(f"Applying LoRA adapters from: {SAVED_MODEL_PATH}")
    model = PeftModel.from_pretrained(
        base_model, SAVED_MODEL_PATH,
        low_cpu_mem_usage=True, device_map="auto",
    )

    # Sovrascriviamo il tokenizer con quello salvato col checkpoint
    tokenizer = AutoTokenizer.from_pretrained(SAVED_MODEL_PATH)
    print("Model and adapters loaded.")

model.eval()
print(f"Model in eval mode. Device: {model.device}")

In [ ]:
# ============================================================
# CELL 5 — Inference function
# ============================================================

def predict_clarity(question: str, answer: str) -> str:
    """
    Predict the clarity macro-category for a question-answer pair.
    Returns one of: 'Clear Reply', 'Ambivalent', 'Clear Non-Reply'.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_TASK1},
        {"role": "user", "content": f"Question: {question}\nPolitician answer: {answer}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# Quick sanity check
print("--- Sanity check (3 examples) ---")
for i in range(min(3, len(test_dataset))):
    es = test_dataset[i]
    pred = predict_clarity(es['interview_question'], es['interview_answer'])
    true_label = str(es.get('clarity_label', '')).strip()
    status = 'OK' if pred == true_label else 'MISS'
    print(f"  [{i+1}] Pred: {pred:20s} | True: {true_label:20s} {status}")

In [ ]:
# ============================================================
# CELL 6 — Performance analysis
# ============================================================

y_true, y_pred = [], []

print(f"Evaluating {len(test_dataset)} examples...")

for i in tqdm(range(len(test_dataset))):
    example = test_dataset[i]
    y_true.append(str(example.get('clarity_label', '')).strip())

    try:
        pred = predict_clarity(example['interview_question'], example['interview_answer'])
        y_pred.append(pred)
    except Exception as e:
        print(f"Error on example {i}: {e}")
        y_pred.append("Error")

# Classification report with three decimal digits
print("\n" + "=" * 60)
print("FINAL REPORT — TASK 1 (CLARITY CLASSIFICATION)")
print("=" * 60)
print(classification_report(y_true, y_pred, labels=LABELS, digits=3))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('LLM predictions')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Task 1 (Direct Clarity)')
plt.tight_layout()
plt.show()

# Save confusion-matrix plot
import os
os.makedirs(RES_DIR, exist_ok=True)
plot_path = f"{TASK1_DIR}/matrice_confusione_task1.png"
plt.savefig(plot_path, bbox_inches='tight', dpi=300)
print(f"\nPlot saved to: {plot_path}")

plt.show()  # Display the plot after saving it

# Save predictions to CSV
print(f"Saving tabular results to: {TASK1_DIR}...")
results_df = pd.DataFrame({
    'Question': [ex.get('interview_question', '') for ex in test_dataset],
    'Politician_Answer': [ex.get('interview_answer', '') for ex in test_dataset],
    'True_Task1_Clarity': y_true,
    'Predicted_Task1_Clarity': y_pred
})

csv_output_path = f"{TASK1_DIR}/predizioni_test_set_task1.csv"
results_df.to_csv(csv_output_path, index=False)
print(f"CSV saved successfully: {csv_output_path}")

In [ ]:
# ============================================================
# CELL 7 — Metrics from CSV (Task 1)
# ============================================================

import os
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

csv_path = os.path.join(TASK1_DIR, "predizioni_test_set_task1.csv")

if not os.path.exists(csv_path):
    print(f"CSV not found: {csv_path}")
else:
    df = pd.read_csv(csv_path)
    if {"Vero_Task1_Clarity", "Predetto_Task1_Clarity"}.issubset(df.columns):
        y_true = df["Vero_Task1_Clarity"].astype(str).str.strip()
        y_pred = df["Predetto_Task1_Clarity"].astype(str).str.strip()

        print("\n" + "=" * 60)
        print("FINAL REPORT — TASK 1 (CSV)")
        print("=" * 60)
        print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
        print(classification_report(y_true, y_pred, labels=LABELS, digits=3))
    else:
        print("Missing CSV columns: Vero_Task1_Clarity, Predetto_Task1_Clarity")